# 01 - Exploratory Data Analysis

SMARD German electricity market data for day-ahead price forecasting.

This notebook inspects shapes, dtypes, missing values, the target
distribution, seasonality, and price drivers - and checks which columns are
safe to use as forecast inputs (leakage analysis).

In [ ]:
%matplotlib inline

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from energy_price_mlops.data.smard import build_smard_dataset

pd.set_option("display.max_columns", 50)

## Load the full 2024 dataset

`build_smard_dataset` merges the three raw SMARD exports on the monotonic
`interval_id`, keeps the local `timestamp`, and adds calendar features.

In [ ]:
raw = Path("../data/raw")
dataset = build_smard_dataset(
    actual_consumption_path=raw / "Actual_consumption_202401010000_202501010000_Hour.csv",
    actual_generation_path=raw / "Actual_generation_202401010000_202501010000_Hour.csv",
    day_ahead_prices_path=raw / "Day-ahead_prices_202401010000_202501010000_Hour.csv",
)
print(f"shape: {dataset.shape}")
dataset.head()

## Schema and dtypes

In [ ]:
dataset.info()

## Missing values

The DE/AT/LU bidding zone was split in 2018, so `price_de_at_lu_eur_mwh` is
expected to be entirely missing for 2024.

In [ ]:
missing = dataset.isna().sum()
missing.to_frame("missing_count").query("missing_count > 0")

## Target distribution

`day_ahead_price_eur_mwh` is the forecast target. Note the negative-price
hours - these occur when renewable generation outruns demand.

In [ ]:
price = dataset["day_ahead_price_eur_mwh"]
print(price.describe())
print(f"\nnegative-price hours: {int((price < 0).sum())}")
print(f"zero-price hours: {int((price == 0).sum())}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
price.plot.hist(bins=60, ax=axes[0], color="steelblue")
axes[0].set_title("Day-ahead price distribution")
axes[0].set_xlabel("EUR/MWh")
axes[1].plot(dataset["timestamp"], price, linewidth=0.6, color="steelblue")
axes[1].set_title("Day-ahead price across 2024")
axes[1].set_xlabel("timestamp")
axes[1].set_ylabel("EUR/MWh")
fig.tight_layout()
plt.show()

## Seasonality

Electricity prices follow a strong daily profile and differ between weekdays
and weekends and across months.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
dataset.groupby("hour")["day_ahead_price_eur_mwh"].mean().plot(
    ax=axes[0], marker="o", color="steelblue"
)
axes[0].set_title("Mean price by hour of day")
axes[0].set_xlabel("hour")
axes[0].set_ylabel("EUR/MWh")
dataset.groupby("day_of_week")["day_ahead_price_eur_mwh"].mean().plot.bar(
    ax=axes[1], color="steelblue"
)
axes[1].set_title("Mean price by day of week (0=Mon)")
dataset.groupby("month")["day_ahead_price_eur_mwh"].mean().plot.bar(
    ax=axes[2], color="steelblue"
)
axes[2].set_title("Mean price by month")
fig.tight_layout()
plt.show()

## Price drivers

Correlation of the target with load and generation columns. Residual load is
expected to push prices up; wind and solar generation push them down (the
merit-order effect).

In [ ]:
drivers = [
    "day_ahead_price_eur_mwh",
    "consumption_grid_load_mwh",
    "consumption_residual_load_mwh",
    "generation_wind_onshore_mwh",
    "generation_wind_offshore_mwh",
    "generation_photovoltaics_mwh",
    "generation_fossil_gas_mwh",
]
dataset[drivers].corr()["day_ahead_price_eur_mwh"].sort_values()

## DST fallback hour

SMARD local timestamps repeat during the Europe/Berlin DST fallback
(`2024-10-27 02:00` appears twice). `interval_id` keeps these rows distinct.

In [ ]:
duplicated_hours = dataset["timestamp"].duplicated(keep=False)
print(f"rows sharing a local timestamp: {int(duplicated_hours.sum())}")
dataset.loc[
    duplicated_hours,
    ["interval_id", "timestamp", "day_ahead_price_eur_mwh"],
]

## Leakage checks

Three groups of columns, and only one is safe as a model **input**:

- **Calendar features** (`hour`, `day_of_week`, `month`, `is_weekend`) are
  deterministic and known in advance - safe.
- **Neighbour-country prices** (`price_*`) are settled in the *same* day-ahead
  auction as the target. Using them to "predict" the DE/LU price is leakage.
- **Actual load and generation** (`consumption_*`, `generation_*`) are
  *realized* values, known only after the fact. They leak unless replaced by
  forecasts or by lagged values.

So the honest forecasting inputs are calendar features plus **lagged** target
and driver values (see `features.add_lag_features`). This constraint shapes the
Phase 2 model.

In [ ]:
target = "day_ahead_price_eur_mwh"
calendar_features = ["hour", "day_of_week", "month", "is_weekend"]
neighbour_prices = [c for c in dataset.columns if c.startswith("price_")]
realized_drivers = [
    c for c in dataset.columns if c.startswith(("consumption_", "generation_"))
]

print(f"target column: {target}")
print(f"safe calendar features ({len(calendar_features)}): {calendar_features}")
print(f"contemporaneous neighbour prices ({len(neighbour_prices)}) - leaky as inputs")
print(f"realized load/generation ({len(realized_drivers)}) - leaky unless lagged")

## Summary

- 8,784 hourly rows for 2024, 39 columns; one local timestamp repeats during
  the Europe/Berlin DST fallback hour - handled by the monotonic `interval_id`.
- `price_de_at_lu_eur_mwh` is fully missing (the DE/AT/LU bidding zone was split
  in 2018) and should be dropped.
- Prices show a strong daily profile, a weekday/weekend gap, and seasonal
  variation; negative prices occur in hours of high renewable generation.
- Residual load correlates positively with price; wind and solar generation
  correlate negatively - the expected merit-order effect.
- Only calendar and lagged features are safe inputs; contemporaneous prices and
  realized load/generation would leak.